In [ ]:
# Install required libraries
pip install -q torch transformers accelerating sympy matplotlib seaborn

## Part 1: Reward Engine

The reward function in DeepSeek-R1-Zero is fully deterministic and rule-based, following:

$$
R_{\text{rule}} = R_{\text{acc}} + R_{\text{fmt}}
$$

To prevent zero-gradient deadlocks in small models, the engine uses a four-tier reward hierarchy:

```text
Generated Completion
├── Format Verification
│   ├── Check <think>...</think>
│   ├── Check <answer>...</answer>
│   └── Score: R_fmt in [0.0, 0.1]
└── Accuracy Verification
    ├── Extract answer string
    ├── Parse with SymPy / Regex
    └── Score: R_acc in [0.0, 1.0]
```

### Reward details

- Strict format match: $R_{\text{fmt}} = 0.1$
- Partial or soft format credit: $R_{\text{fmt}} \in (0.0, 0.1)$
- Accuracy match via SymPy: $R_{\text{acc}} = 1.0$
- Fallback raw string match if parsing fails


In [1]:
import subprocess
import sys
from pathlib import Path

repo_dir = Path.cwd()
engine_path = repo_dir / "reward_engine.py"

if not engine_path.exists():
    raise FileNotFoundError(f"Could not find reward_engine.py at {engine_path}")

result = subprocess.run(
    [sys.executable, str(engine_path)],
    cwd=repo_dir,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)


=== Running Unit Test Suite for reward_engine.py ===
Test 1 [Perfect Output (Correct + Strict Format)]: PASSED
  Result -> Reward: 1.100 | Correct: True | Extracted: '30'
Test 2 [SymPy Equivalence (Fraction vs Decimal)]: PASSED
  Result -> Reward: 1.100 | Correct: True | Extracted: '0.5'
Test 3 [Wrong Answer + Strict Format]: PASSED
  Result -> Reward: 0.100 | Correct: False | Extracted: '99'
Test 4 [Partial Tags (Soft Format Credit)]: PASSED
  Result -> Reward: 0.025 | Correct: False | Extracted: '<think> 14 * 3 = 42, Answer is 30'
Test 5 [No Tags At All]: PASSED
  Result -> Reward: 0.000 | Correct: False | Extracted: 'The result is 30'

Unit Test Suite Summary: ALL TESTS PASSED!



## Dataset

In the DeepSeek-R1 paper, training requires framing prompts with a strict conversational structure that instructs the model to place its reasoning inside `<think>...</think>` tags and its final solution inside `<answer>...</answer>` tags.

For the arithmetic tasks, the dataset module will:

- programmatically generate multi-step arithmetic problems,
- provide mathematical ground truths,
- wrap each problem in the R1-Zero prompt template,
- and prime the generation head by appending `<think>` so the model begins reasoning immediately during rollouts.


In [ ]:
import subprocess
import sys
from pathlib import Path

repo_dir = Path.cwd()
engine_path = repo_dir / "dataset.py"

if not engine_path.exists():
    raise FileNotFoundError(f"Could not find reward_engine.py at {engine_path}")

result = subprocess.run(
    [sys.executable, str(engine_path)],
    cwd=repo_dir,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)


=== Running Unit Test Suite for dataset.py ===

Sample 1:
  Expression:   12*3-1=
  Ground Truth: 35
  Full Prompt:
A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think>...</think> and <answer>...</answer> tags, respectively. User: Solve 12*3-1= Assistant: <think>

Sample 2:
  Expression:   5*5+5=
  Ground Truth: 30
  Full Prompt:
A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think>...</think> and <answer>...</answer> tags, respectively. User: Solve 5*5+5= Assistant: <think>

Sample 3:
  Expression:   12*10-3=
  Ground Truth: 117
  Full Prompt:
A conversation b

## Loss

### Mathematical review and paper alignment

- Group advantage calculation ($A_i$, Equation 3):

$$
A_i = \frac{r_i - \text{mean}(\mathbf{r})}{\text{std}(\mathbf{r}) + \epsilon}
$$

- If $\text{std}(\mathbf{r}) = 0$, then $A_i$ is set to $0.0$ for the full group to avoid division by zero.

- Unbiased token-level KL divergence ($D_{\text{KL}}$, Equation 2):

$$
D_{\text{KL}}(\pi_\theta \;\|\|\; \pi_{\text{ref}})
=
\frac{\pi_{\text{ref}}(y_t \mid x, y_{<t})}{\pi_\theta(y_t \mid x, y_{<t})}
-
\log\frac{\pi_{\text{ref}}(y_t \mid x, y_{<t})}{\pi_\theta(y_t \mid x, y_{<t})}
- 1
$$

- Using log-probabilities directly, where $\Delta = \log \pi_{\text{ref}}(y_t) - \log \pi_\theta(y_t)$:

$$
D_{\text{KL}} = \exp(\Delta) - \Delta - 1
$$

- Clipped surrogate loss ($\mathcal{J}_{\text{GRPO}}$, Equation 1):

$$
\mathcal{L}_{\text{surr}} = \min \left( r_{i,t}(\theta) A_i, \; \text{clip}(r_{i,t}(\theta), 1-\epsilon, 1+\epsilon) A_i \right)
$$

$$
\mathcal{L}_{\text{total}} = -\frac{1}{G \cdot T} \sum_{i=1}^{G} \sum_{t=1}^{T} \left( \mathcal{L}_{\text{surr}, i, t} - \beta \cdot D_{\text{KL}, i, t} \right)
$$


In [1]:
import subprocess
import sys
from pathlib import Path

repo_dir = Path.cwd()
engine_path = repo_dir / "grpo_loss.py"

if not engine_path.exists():
    raise FileNotFoundError(f"Could not find reward_engine.py at {engine_path}")

result = subprocess.run(
    [sys.executable, str(engine_path)],
    cwd=repo_dir,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)
